# This might be all we need!

In [1]:
from schwingermodel import generateSchwingerHamiltonian

In [2]:
from qiskit.quantum_info import SparsePauliOp

def bitflip_projection(op: SparsePauliOp) -> SparsePauliOp:
    terms = []
    for label, coeff in zip(op.paulis.to_labels(), op.coeffs):
        new_label = label.replace("Y", "X").replace("Z", "I")
        terms.append((new_label, coeff))
    return SparsePauliOp.from_list(terms).simplify()


def apply_ix_string_to_bitstring(ix_label: str, bitstring: str) -> str:
    out = list(bitstring)
    for j, p in enumerate(ix_label):
        if p == "X":
            out[j] = "1" if out[j] == "0" else "0"
    return "".join(out)


def reachable_bitstrings_n_steps(op: SparsePauliOp, initial_bitstrings, n_steps: int):
    """
    Ignore amplitudes entirely.
    Track only which bitstrings are reachable after n_steps applications
    of the projected operator's flip masks.
    """
    if isinstance(initial_bitstrings, str):
        current = {initial_bitstrings}
    else:
        current = set(initial_bitstrings)

    labels = op.paulis.to_labels()

    for _ in range(n_steps):
        nxt = set()
        for b in current:
            for label in labels:
                nxt.add(apply_ix_string_to_bitstring(label, b))
        current = nxt

    return current

In [3]:
# Example parameters
N = 4
x = 1.0
lam = 0.5
l0 = 0.0
m_lat = 1.0
g = 1.0

In [4]:
N = 40
H = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)
H_flip = bitflip_projection(H)

initial_state = '01' * (N//2)

reachable = reachable_bitstrings_n_steps(H_flip, initial_state, 3)
print(len(reachable))

9920


# Post-processing

# does not work yet

In [6]:
counts_all = d = {bitstring: 1 for bitstring in reachable}

In [7]:
from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.qubit import solve_qubit
from qiskit_addon_sqd.qubit import sort_and_remove_duplicates
from qiskit_addon_sqd.qubit import project_operator_to_subspace

from schwingermodel import WS_sparse_pauli_op

import numpy as np
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import eigsh

In [ ]:
def get_proj_ham(Ham, counts):
    # for idx, counts in enumerate(tqdm.tqdm(counts_cumulative, desc="Processing counts")):
    # counts = counts_cumulative[1]
    # counts = postselect_counts(counts, num_ones=Ham.num_qubits // 2)
    bitstring_matrix, probs = counts_to_arrays(counts=counts)

    bitstrings_sorted = sort_and_remove_duplicates(bitstring_matrix)
    return project_operator_to_subspace(bitstrings_sorted, Ham), project_operator_to_subspace(bitstrings_sorted, Ls), project_operator_to_subspace(bitstrings_sorted, Ps)

: 

In [ ]:
thetas_exact = np.linspace(2.5, 3.5, 1000)
energies_exact = np.zeros(len(thetas_exact))
ls_exact = np.zeros(len(thetas_exact))
ps_exact = np.zeros(len(thetas_exact))
Hams = []

for i, theta in enumerate((thetas_exact)):
    Hamiltonian = WS_sparse_pauli_op(N=20, x=(N/30)**2, lam=1e3, l0=theta, m_lat=10, g=1)
    Hams.append(Hamiltonian)
    Hamiltonian_matrix = csc_matrix(Hamiltonian.to_matrix())
    evals, evecs = eigsh(Hamiltonian_matrix, k=1, which='SA')
    energies_exact[i] = evals

In [ ]:
for ham in Hams: 
    H_proj, L_proj, P_proj = get_proj_ham(ham, counts_all)

    eigvals, eigvecs = eigh(H_proj.toarray())

    gs_en = np.min(eigvals)
    gs_index = np.argmin(eigvals)

    ls_two_dim.append(np.real(eigvecs[gs_index].conj().T @ L_proj @ eigvecs[gs_index]))
    ps_two_dim.append(np.real(eigvecs[gs_index].conj().T @ P_proj @ eigvecs[gs_index]))



    energies_two_dim.append(gs_en)

AttributeError: 'set' object has no attribute 'values'